In [1]:
import pandas as pd
from pathlib import Path

In [2]:
file_path = Path("..") / "data" / "interim" / "transactions_merged.csv"
df = pd.read_csv(file_path)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [4]:
df_cleaned = df.dropna(subset=["Description", "Customer ID"]).reset_index(drop=True)

In [5]:
df_cleaned["InvoiceDate"] = pd.to_datetime(df_cleaned["InvoiceDate"])

In [6]:
df_cleaned["InvoiceDate"].dtype

dtype('<M8[us]')

In [7]:
df_cleaned["is_cancelled"] = df_cleaned["Invoice"].astype(str).str.startswith("C")

In [19]:
# creating two copies
df_purchases = df_cleaned[~df_cleaned["is_cancelled"]].copy()
df_cancellations = df_cleaned[df_cleaned["is_cancelled"]].copy()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancelled
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,True
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True


In [20]:
max_date = df_purchases["InvoiceDate"].max()

line = max_date - pd.DateOffset(months=1)

df_purchase_train = df_purchases[df_purchases["InvoiceDate"] < line].copy()
df_purchase_test = df_purchases[df_purchases["InvoiceDate"] >= line].copy()
df_cancellations_train = df_cancellations[df_cancellations["InvoiceDate"] < line].copy() 
df_cancellations_test = df_cancellations[df_cancellations["InvoiceDate"] >= line].copy() 

In [41]:
df_purchase_test.info()

<class 'pandas.DataFrame'>
Index: 65582 entries, 757731 to 824363
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Invoice       65582 non-null  str           
 1   StockCode     65582 non-null  str           
 2   Description   65582 non-null  str           
 3   Quantity      65582 non-null  int64         
 4   InvoiceDate   65582 non-null  datetime64[us]
 5   Price         65582 non-null  float64       
 6   Customer ID   65582 non-null  float64       
 7   Country       65582 non-null  str           
 8   is_cancelled  65582 non-null  bool          
dtypes: bool(1), datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 4.6 MB


In [22]:
# creating customer basic features - train
df_purchase_train["TotalPrice"] = df_purchase_train["Quantity"] * df_purchase_train["Price"]

customer_features_train = df_purchase_train.groupby("Customer ID").agg(
    unique_products=("StockCode", "nunique"),
    total_quantity=("Quantity", "sum"),
    last_purchase=("InvoiceDate", "max"),
    total_orders=("Invoice", "nunique"),
    total_spent=("TotalPrice", "sum"),
    country=("Country", "first")
).reset_index()

df_train = pd.concat([df_purchase_train, df_cancellations_train], axis=0)

customer_cancellations_train = df_train.groupby("Customer ID").agg(
    cancelled_lines_count=("is_cancelled", "sum"),
    total_lines=("Invoice", "size")
).reset_index()

customer_cancellations_train["cancellation_ratio"] = (
    customer_cancellations_train["cancelled_lines_count"] / customer_cancellations_train["total_lines"]
)

customer_features_train = customer_features_train.merge(
    customer_cancellations_train[["Customer ID", "cancelled_lines_count", "cancellation_ratio"]],
    on="Customer ID",
    how="left"
)

customer_features_train["cancelled_lines_count"] = customer_features_train["cancelled_lines_count"].fillna(0)
customer_features_train["cancellation_ratio"] = customer_features_train["cancellation_ratio"].fillna(0)

In [23]:
reference_date = customer_features_train["last_purchase"].max() + pd.Timedelta(days=1)

customer_features_train["days_since_last_purchase"] = (
    reference_date - customer_features_train["last_purchase"]
).dt.days

In [25]:
customer_features_train = customer_features_train.drop(columns=["last_purchase"])

In [35]:
# adding some additional data so model would be able to predict actions - train

orders_train = (
    df_purchase_train.groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(order_date=("InvoiceDate", "max"))
)

orders_train = orders_train.sort_values(["Customer ID", "order_date"])

orders_train["days_between_orders"] = (
    orders_train.groupby("Customer ID")["order_date"]
    .diff()
    .dt.days
)

avg_gap_train = (
    orders_train.groupby("Customer ID")["days_between_orders"]
    .mean()
    .reset_index(name="avg_days_between_orders")
)

customer_features_train = customer_features_train.merge(
    avg_gap_train[["Customer ID", "avg_days_between_orders"]],
    on="Customer ID",
    how="left"
)

customer_time_train = df_purchase_train.groupby("Customer ID").agg(
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

customer_time_train["customer_lifetime_days"] = (
    customer_time_train["last_purchase"] - customer_time_train["first_purchase"]
).dt.days

customer_features_train = customer_features_train.merge(
    customer_time_train[["Customer ID", "customer_lifetime_days"]],
    on="Customer ID",
    how="left"
)

In [37]:
customer_features_train.head()

,Customer ID,unique_products,total_quantity,total_orders,total_spent,country,cancelled_lines_count,cancellation_ratio,days_since_last_purchase,avg_days_between_orders,customer_lifetime_days,is_short_history,orders_per_month
0,12346.0,27,74285,12,77556.46,United Kingdom,14,0.291667,296,35.909091,400,0,0.900000
1,12347.0,123,3094,7,5408.50,Iceland,0,0.000000,10,60.333333,364,0,0.576923
2,12348.0,25,2714,5,2019.40,Finland,0,0.000000,45,90.500000,362,0,0.414365
3,12349.0,90,993,3,2671.14,Italy,5,0.046729,378,90.000000,181,0,0.497238
4,12350.0,17,197,1,334.40,Norway,0,0.000000,280,-1.000000,0,1,0.000000


In [36]:
customer_features_train["is_short_history"] = (
    customer_features_train["customer_lifetime_days"] < 30
).astype(int)

customer_features_train["orders_per_month"] = customer_features_train.apply(
    lambda row: 0
    if row["customer_lifetime_days"] < 30
    else row["total_orders"] / (row["customer_lifetime_days"] / 30),
    axis=1
)

customer_features_train["avg_days_between_orders"] = (
    customer_features_train["avg_days_between_orders"].fillna(-1)
)

In [42]:
# preparing  answers for model 
# 1 -> if client bought and didn't cancel
# 0 -> client bought and cancelled or didn't buy

buyers_id = df_purchase_test["Customer ID"].dropna().unique()

eval_data = customer_features_train.copy()

eval_data["made_purchase_next_30d"] = (
    eval_data["Customer ID"].isin(buyers_id)
).astype(int)

In [44]:
customer_features_train = customer_features_train.merge(
    eval_data[["Customer ID", "made_purchase_next_30d"]],
    on="Customer ID",
    how="left"
)

In [54]:
print(customer_features_train["made_purchase_next_30d"].value_counts())
print(customer_features_train["made_purchase_next_30d"].value_counts(normalize=True))

made_purchase_next_30d
0    4232
1    1487
Name: count, dtype: int64
made_purchase_next_30d
0    0.73999
1    0.26001
Name: proportion, dtype: float64


In [56]:
out_path = Path("..") / "data" / "processed" / "prepared_data.csv"
customer_features_train.to_csv(out_path, index=False)